# DeepAgents Harness 빈칸 채우기 실습

이 노트북에서는 Ch3 Step 1~2에서 배운 **DeepAgents의 핵심 구성**을 직접 채워 봅니다.

**실습 방법:**
- 코드 셀의 `___` 부분이 빈칸입니다.
- 힌트를 참고하여 알맞은 코드를 채워 넣으세요.
- 모든 빈칸을 채운 뒤 셀을 실행하면 동작을 확인할 수 있습니다.
- 노트북 맨 하단에 정답이 있습니다.

| 실험 | 내용 | 빈칸 | 시간 |
|------|------|------|------|
| 실험 1 | `create_deep_agent()` 구성 + 동작 확인 | 3개 | 5분 |
| 실험 2 | `FilesystemBackend` 설정 | 3개 | 5분 |

In [ ]:
# 공통 import
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain_core.tools import tool
from pathlib import Path

# 커스텀 Tool 정의 (실험 1에서 사용)
@tool
def check_inbox(filter: str = "unread") -> str:
    """메일함의 메일 목록을 확인합니다."""
    return f"{filter} 메일 3통 확인됨"

print("import 완료")

## 실험 1: `create_deep_agent()` 구성 (5분)

Ch2에서는 StateGraph + 노드 + 엣지를 직접 정의했지만,
DeepAgents는 `create_deep_agent()` **한 줄**로 Agent를 구성합니다.

세 가지 핵심 파라미터를 채워 Agent를 생성하세요:
- `model`: LLM 모델 식별자 (`"openai:google/gemini-3-flash-preview"`)
- `tools`: 커스텀 Tool 리스트
- `system_prompt`: Agent의 역할 지시문

In [ ]:
# ===== 빈칸을 채우세요 =====

agent = create_deep_agent(
    model=___,           # 힌트: "openai:" 프리픽스 + 모델 이름 (문자열)
    tools=[___],         # 힌트: 위에서 정의한 Tool 함수 이름
    system_prompt=___,   # 힌트: Agent 역할을 설명하는 문자열
)

# Agent 생성 확인
# 참고: 내장 Tool(write_todos, read_file 등)은 미들웨어 내부에 등록되어
#       agent.tools로 직접 접근할 수 없습니다.
#       실제 동작은 agent.invoke()로 확인합니다.
print(f"Agent 타입: {type(agent).__name__}")
print(f"Agent 생성 완료")

# 간단한 동작 테스트 — Tool 호출 없이 응답만 확인
test_result = agent.invoke({"messages": [{"role": "user", "content": "안녕하세요. 간단히 인사만 해주세요."}]})
last_msg = test_result["messages"][-1]
assert hasattr(last_msg, "content"), "Agent 응답이 없습니다 — create_deep_agent() 파라미터를 확인하세요"
print(f"Agent 응답: {last_msg.content[:100]}")
print("\n실험 1 통과: Agent 구성 + 동작 확인 완료")

## 실험 2: `FilesystemBackend` 설정 (5분)

DeepAgents의 기본 Backend는 **StateBackend**(인메모리)입니다.
`FilesystemBackend`로 바꾸면 Agent가 생성한 파일이 **실제 디스크**에 저장됩니다.

세 가지 빈칸을 채우세요:
- `root_dir`: 파일이 저장될 디렉토리 경로
- `virtual_mode`: 디렉토리 탈출 방지 보안 옵션 (`True`/`False`)
- `backend`: Agent 생성 시 backend 파라미터에 전달할 객체

In [ ]:
# ===== 빈칸을 채우세요 =====

WORKSPACE = Path("./experiment_workspace")
WORKSPACE.mkdir(parents=True, exist_ok=True)

fs_backend = FilesystemBackend(
    root_dir=___,        # 힌트: WORKSPACE를 문자열로 변환 (str(...))
    virtual_mode=___,    # 힌트: 보안을 위해 디렉토리 탈출 방지 (True 또는 False)
)

fs_agent = create_deep_agent(
    model="openai:google/gemini-3-flash-preview",
    backend=___,         # 힌트: 위에서 만든 백엔드 객체
)

print(f"FilesystemBackend 설정 완료")
print(f"  root_dir: {WORKSPACE.resolve()}")
print(f"  Agent가 write_file() 호출 시 → {WORKSPACE.resolve()}/ 아래에 저장됨")
print("\n실험 2 통과: FilesystemBackend 구성 완료")

## 핵심 비교

| 기능 | Ch2 (수동) | DeepAgents (내장) |
|------|-----------|------------------|
| 파일 쓰기 | `save_to_file` 직접 구현 (~15줄) | `write_file` 내장 |
| 파일 읽기 | `read_from_file` 직접 구현 (~15줄) | `read_file` 내장 |
| 작업 계획 | `write_todos` 직접 구현 (~30줄) | `write_todos` 미들웨어 자동 제공 |
| 하위 작업 위임 | 미구현 | `task` Tool로 SubAgent 자동 위임 |
| 그래프 구성 | StateGraph ~30줄 | `create_deep_agent()` 1줄 |
| 파일 저장 방식 | 경로 하드코딩 | Backend 교체로 유연하게 전환 |

In [ ]:
# 전체 실험 결과 정리
print("=" * 50)
print("전체 실험 결과")
print("=" * 50)

print("\n[실험 1] create_deep_agent() 구성")
print("  - model, tools, system_prompt 3개 파라미터로 Agent 생성")
print("  - 내장 Tool 9개 자동 포함 (미들웨어 내부에 등록)")

print("\n[실험 2] FilesystemBackend 설정")
print("  - root_dir + virtual_mode로 디스크 저장 구성")
print("  - Ch2의 save_to_file 30줄 → backend 파라미터 1줄")

print("\n모든 실험 통과!")

## 학습 포인트

1. **`create_deep_agent(model, tools, system_prompt)`** — 이 세 파라미터가 Agent 구성의 핵심입니다.
2. **`FilesystemBackend(root_dir, virtual_mode)`** — Backend를 교체하면 저장 방식이 바뀝니다. Agent 코드는 변경 불필요.
3. **내장 Tool 9개**가 미들웨어를 통해 자동 등록됩니다: `write_todos`, `read_file`, `write_file`, `edit_file`, `ls`, `glob`, `grep`, `execute`, `task`
4. Ch2에서 ~110줄로 만든 보일러플레이트가 `create_deep_agent()` + `FilesystemBackend` 조합으로 대체됩니다.

---

## 정답

<details>
<summary><b>클릭하여 정답 확인</b></summary>

### 실험 1: `create_deep_agent()` 구성

```python
agent = create_deep_agent(
    model="openai:google/gemini-3-flash-preview",
    tools=[check_inbox],
    system_prompt="당신은 메일 관리 비서 Agent입니다.",
)
```

**해설:**
- `model`: `"openai:"` 프리픽스는 OpenAI 호환 API(OpenRouter 포함)를 뜻합니다.
- `tools`: 위에서 `@tool`로 정의한 `check_inbox` 함수를 리스트로 전달합니다. 내장 Tool은 미들웨어 내부에 자동 추가됩니다.
- `system_prompt`: Agent의 역할을 설명하는 문자열입니다.

---

### 실험 2: `FilesystemBackend` 설정

```python
fs_backend = FilesystemBackend(
    root_dir=str(WORKSPACE),
    virtual_mode=True,
)

fs_agent = create_deep_agent(
    model="openai:google/gemini-3-flash-preview",
    backend=fs_backend,
)
```

**해설:**
- `root_dir=str(WORKSPACE)`: Path 객체를 문자열로 변환하여 전달합니다.
- `virtual_mode=True`: Agent가 root_dir 바깥 경로에 접근하는 것을 차단합니다 (보안).
- `backend=fs_backend`: 생성한 FilesystemBackend 객체를 Agent에 전달합니다.

</details>